# 🌱 AgriPipe — Demo end-to-end

Tour guidato della pipeline: dalla generazione di un Excel sporco alla produzione di un tensor PyTorch pronto per il training.

**Pipeline:**
1. Genera Excel sintetico con anomalie realistiche
2. Caricamento + validazione schema
3. Pulizia automatica
4. Report HTML di qualità
5. Conversione in `torch.Tensor` + `DataLoader`

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from IPython.display import IFrame
from torch.utils.data import DataLoader

from agripipe import (
    AgriCleaner,
    AgriDataset,
    SynthConfig,
    generate_dirty_excel,
    generate_report,
    load_raw,
)

OUT = Path("../out")
OUT.mkdir(exist_ok=True)

## 1. Genera un Excel sporco sintetico

In [ ]:
xlsx_path = OUT / "synthetic_dirty.xlsx"
generate_dirty_excel(xlsx_path, SynthConfig(n_rows=500, seed=42))

df_raw = pd.read_excel(xlsx_path)
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Panoramica anomalie iniettate
print("NaN per colonna:")
print(df_raw.isna().sum())
print(f"\nDuplicati: {df_raw.duplicated().sum()}")
print(f"\npH fuori range [0,14]: {((df_raw['ph'] < 0) | (df_raw['ph'] > 14)).sum()}")
print(f"Humidity > 100%: {(df_raw['humidity'] > 100).sum()}")

## 2. Caricamento con validazione

In [ ]:
df_raw = load_raw(xlsx_path)

## 3. Pulizia con config YAML

In [ ]:
cleaner = AgriCleaner.from_yaml("../configs/default.yaml")
df_clean = cleaner.clean(df_raw)

print(f"Righe: {len(df_raw)} → {len(df_clean)}")
print(f"NaN residui: {df_clean.isna().sum().sum()}")
df_clean.describe().round(2)

## 4. Report HTML di qualità

In [ ]:
report_path = generate_report(df_raw, df_clean, OUT / "quality_report.html")

IFrame(str(report_path), width="100%", height=600)

## 5. Conversione in tensor PyTorch

In [ ]:
ds = AgriDataset(
    df_clean,
    numeric_columns=cleaner.config.numeric_columns,
    categorical_columns=cleaner.config.categorical_columns,
    target="yield",
)

print(f"Dataset size: {len(ds)}")
print(f"Features shape: {ds.features.shape}")
print(f"Target shape:   {ds.target.shape}")
print(f"Feature names:  {ds.feature_names}")

## 6. DataLoader pronto per il training

In [ ]:
loader = DataLoader(ds, batch_size=32, shuffle=True)

X_batch, y_batch = next(iter(loader))
print(f"Batch features: {X_batch.shape} ({X_batch.dtype})")
print(f"Batch target:   {y_batch.shape} ({y_batch.dtype})")
print(f"\nSample:\n{X_batch[0]}")

## 7. Esempio minimo di training loop

Solo per dimostrare che il tensor è effettivamente utilizzabile. Non è un modello serio.

In [ ]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(ds.features.shape[1], 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(3):
    total = 0.0
    for X, y in loader:
        opt.zero_grad()
        pred = model(X).squeeze()
        loss = loss_fn(pred, y)
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"Epoch {epoch+1} — loss: {total/len(loader):.4f}")

---
✅ Pipeline completa: **Excel sporco → tensor PyTorch → training loop funzionante.**